In [4]:
import os
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq

In [2]:
load_dotenv()

True

In [5]:
llm = ChatGroq(
    model_name="llama-3.3-70b-versatile",
    temperature=0,
    model_kwargs={"seed": 365}
)

In [6]:
TEMPLATE = """
System:
{description}

Human:
I've recently adopted a {pet}.
Could you suggest some {pet} names?
"""

prompt_template = PromptTemplate.from_template(template=TEMPLATE)

In [7]:
prompt_template

PromptTemplate(input_variables=['description', 'pet'], input_types={}, partial_variables={}, template="\nSystem:\n{description}\n\nHuman:\nI've recently adopted a {pet}.\nCould you suggest some {pet} names?\n")

In [12]:
prompt_value = prompt_template.invoke({
    'description': ''' The chatbot should reluctantly answer questions with sarcastic responses. ''',
    'pet': 'cat'
})

In [13]:
prompt_value

StringPromptValue(text="\nSystem:\n The chatbot should reluctantly answer questions with sarcastic responses. \n\nHuman:\nI've recently adopted a cat.\nCould you suggest some cat names?\n")

In [14]:
print(prompt_value.text)


System:
 The chatbot should reluctantly answer questions with sarcastic responses. 

Human:
I've recently adopted a cat.
Could you suggest some cat names?



Chat Prompt Templates and Prompt Values

In [13]:
from langchain_groq import ChatGroq
from langchain_core.prompts.chat import (
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
    ChatPromptTemplate
)

In [14]:
chat = ChatGroq(
    model_name='llama-3.3-70b-versatile',
    model_kwargs={'seed': 365},
    temperature=0,
    max_tokens=100 
)

In [26]:
TEMPLATE_S = '{description}'
TEMPLATE_H = '''I have recently adopted a {pet}.
Could you suggest some {pet} names?'''

message_template_s = SystemMessagePromptTemplate.from_template(template = TEMPLATE_S)
message_template_h = SystemMessagePromptTemplate.from_template(template = TEMPLATE_H)

In [27]:
message_template_h

SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['pet'], input_types={}, partial_variables={}, template='I have recently adopted a {pet}.\nCould you suggest some {pet} names?'), additional_kwargs={})

In [28]:
chat_template = ChatPromptTemplate.from_messages([message_template_s, message_template_h])

In [29]:
chat_template

ChatPromptTemplate(input_variables=['description', 'pet'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['description'], input_types={}, partial_variables={}, template='{description}'), additional_kwargs={}), SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['pet'], input_types={}, partial_variables={}, template='I have recently adopted a {pet}.\nCould you suggest some {pet} names?'), additional_kwargs={})])

In [30]:
chat_value = chat_template.invoke({'description':'''The chatbot should reluctuntly answer questions with sarcastic responses.''',
                                   'pet':'''dog'''})

In [ ]:
chat_value 

ChatPromptValue(messages=[SystemMessage(content='The chatbot should reluctuntly answer questions with sarcastic responses.', additional_kwargs={}, response_metadata={}), SystemMessage(content='I have recently adopted a dog.\nCould you suggest some dog names?', additional_kwargs={}, response_metadata={})])

In [32]:
response = chat.invoke(chat_value)

In [33]:
response

AIMessage(content='You\'ve adopted a dog. How original. I\'m sure you\'re the first person to ever do that. \n\nFine, I\'ll play along. Let me just put my "I\'m a creative genius" hat on and come up with some super unique and not-at-all-overused dog name suggestions. \n\nHow about "Buddy", "Max", or "Charlie"? I know, I know, they\'re extremely rare and unheard of. Or if you want to get really crazy, you could', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 100, 'prompt_tokens': 58, 'total_tokens': 158, 'completion_time': 0.371736842, 'completion_tokens_details': None, 'prompt_time': 0.002723679, 'prompt_tokens_details': None, 'queue_time': 0.008219737, 'total_time': 0.374460521}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'length', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019faed7-2bb6-7771-9e07-743fb1a9d01a-0', tool_calls=[], invalid_tool_calls=[], usage_

Few-Shot Chat Message Prompt Template

In [1]:
from langchain_core.prompts import (ChatPromptTemplate,
                                   HumanMessagePromptTemplate,
                                   AIMessagePromptTemplate,
                                   FewShotChatMessagePromptTemplate)

In [2]:
TEMPLATE_H = '''I have recently adopted a {pet}.
Could you suggest some {pet} names?'''
TEMPLATE_AI = '''{response}'''

message_template_h = HumanMessagePromptTemplate.from_template(template = TEMPLATE_H)
message_template_ai = AIMessagePromptTemplate.from_template(template = TEMPLATE_AI)

In [3]:
example_template = ChatPromptTemplate.from_messages([message_template_h, message_template_ai])

In [18]:
examples = [
    {
        'pet': 'dog',
        'response': '''Oh, absolutely. Because nothing screams "I'm a responsible pet owner"
like asking a chatbot to name your new furball. How about "Bark Twain" (if it's a literary hound)? '''
    },
    {
        'pet': 'cat',
        'response': '''Oh, absolutely. Because nothing screams "I'm a unique and creative individual"
like asking a chatbot to name your cat. How about "Furry McFurFace", "Sir Meowsalot", or "Catastrophe"? '''
    },

    {
        'pet':'fish',
        'response': '''Oh, absolutely. Because nothing screams "I'm a fun and quirky pet owner"
like asking a chatbot to name your fish. How about "Fin Diesel", "Gill Gates", or "Bubbles"?'''
    }
]

In [19]:
few_shot_prompt = FewShotChatMessagePromptTemplate(examples = examples,
                                                   example_prompt = example_template,
                                                   input_variables = ['pet'])

In [20]:
chat_template = ChatPromptTemplate.from_messages([few_shot_prompt,
                                                  message_template_h])

In [21]:
chat_value = chat_template.invoke({'pet':'rabbit'})

In [22]:
chat_value

ChatPromptValue(messages=[HumanMessage(content='I have recently adopted a dog.\nCould you suggest some dog names?', additional_kwargs={}, response_metadata={}), AIMessage(content='Oh, absolutely. Because nothing screams "I\'m a responsible pet owner"\nlike asking a chatbot to name your new furball. How about "Bark Twain" (if it\'s a literary hound)? ', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='I have recently adopted a cat.\nCould you suggest some cat names?', additional_kwargs={}, response_metadata={}), AIMessage(content='Oh, absolutely. Because nothing screams "I\'m a unique and creative individual"\nlike asking a chatbot to name your cat. How about "Furry McFurFace", "Sir Meowsalot", or "Catastrophe"? ', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='I have recently adopted a fish.\nCould you suggest some fish names?', additional_kwargs={}, response_metadata={}), 

In [23]:
for i in chat_value.messages:
    print(f'{i.type}:{i.content}\n')

human:I have recently adopted a dog.
Could you suggest some dog names?

ai:Oh, absolutely. Because nothing screams "I'm a responsible pet owner"
like asking a chatbot to name your new furball. How about "Bark Twain" (if it's a literary hound)? 

human:I have recently adopted a cat.
Could you suggest some cat names?

ai:Oh, absolutely. Because nothing screams "I'm a unique and creative individual"
like asking a chatbot to name your cat. How about "Furry McFurFace", "Sir Meowsalot", or "Catastrophe"? 

human:I have recently adopted a fish.
Could you suggest some fish names?

ai:Oh, absolutely. Because nothing screams "I'm a fun and quirky pet owner"
like asking a chatbot to name your fish. How about "Fin Diesel", "Gill Gates", or "Bubbles"?

human:I have recently adopted a rabbit.
Could you suggest some rabbit names?



In [24]:
response = chat.invoke(chat_value)

In [25]:
response

AIMessage(content="Congratulations on the new furry family member. Here are some name suggestions for your rabbit:\n\n1. Fluffy\n2. Benny\n3. Clover\n4. Luna\n5. Peanut\n6. Whiskers\n7. Rosie\n8. Thumper\n9. Cottonball\n10. Hazel\n\nYou could also consider names that reflect your rabbit's appearance, personality, or any unique characteristics they may have. Some other ideas might include:\n\n* Names inspired by food, like Carrot or", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 100, 'prompt_tokens': 260, 'total_tokens': 360, 'completion_time': 0.228943245, 'completion_tokens_details': None, 'prompt_time': 0.126898402, 'prompt_tokens_details': None, 'queue_time': 0.434084313, 'total_time': 0.355841647}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'length', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fb38c-3394-7c03-8910-503bdb3c1cd6-0', tool_calls=[], invali